In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest, RandomForestClassifier

In [2]:
dataset = pd.read_csv("wustl-ehms-2020_with_attacks_categories-checkpoint.csv")
df = dataset.copy()
df.head()

,Dir,Flgs,SrcAddr,DstAddr,Sport,Dport,SrcBytes,DstBytes,SrcLoad,DstLoad,...,Temp,SpO2,Pulse_Rate,SYS,DIA,Heart_rate,Resp_Rate,ST,Attack Category,Label
0,->,e,10.0.1.172,10.0.1.150,58059,1111,496,186,276914.0,92305.0,...,28.9,0,0,0,0,0,0,0.0,normal,0
1,->,e,10.0.1.172,10.0.1.150,58062,1111,496,186,230984.0,76995.0,...,28.9,0,0,0,0,78,17,0.4,normal,0
2,->,e,10.0.1.172,10.0.1.150,58065,1111,496,186,218470.0,72823.0,...,28.9,89,104,0,0,78,17,0.4,normal,0
3,->,e,10.0.1.172,10.0.1.150,58067,1111,496,186,203376.0,67792.0,...,28.9,89,104,0,0,79,17,0.4,normal,0
4,->,e,10.0.1.172,10.0.1.150,58069,1111,496,186,235723.0,78574.0,...,28.9,89,101,0,0,79,17,0.4,normal,0


In [3]:
df.columns

Index(['Dir', 'Flgs', 'SrcAddr', 'DstAddr', 'Sport', 'Dport', 'SrcBytes',
       'DstBytes', 'SrcLoad', 'DstLoad', 'SrcGap', 'DstGap', 'SIntPkt',
       'DIntPkt', 'SIntPktAct', 'DIntPktAct', 'SrcJitter', 'DstJitter',
       'sMaxPktSz', 'dMaxPktSz', 'sMinPktSz', 'dMinPktSz', 'Dur', 'Trans',
       'TotPkts', 'TotBytes', 'Load', 'Loss', 'pLoss', 'pSrcLoss', 'pDstLoss',
       'Rate', 'SrcMac', 'DstMac', 'Packet_num', 'Temp', 'SpO2', 'Pulse_Rate',
       'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST', 'Attack Category',
       'Label'],
      dtype='object')

In [4]:
#just analyzing

FEATURE_COLS = ['Dir', 'Flgs', 'Sport', 'Dport',
    'SrcBytes', 'DstBytes',
    'SrcLoad', 'DstLoad',
    'SrcGap', 'DstGap',
    'SIntPkt', 'DIntPkt',
    'SIntPktAct', 'DIntPktAct',
    'SrcJitter', 'DstJitter',
    'sMaxPktSz', 'dMaxPktSz',
    'sMinPktSz', 'dMinPktSz',
    'Dur', 'Trans',
    'TotPkts', 'TotBytes',
    'Load', 'Rate']

LABEL_COL = "Label"
ATTACK_COL = "Attack Category"

X = df[FEATURE_COLS].copy()
y_binary = df[LABEL_COL].copy()
y_multi = df[ATTACK_COL].copy()

print("X shape:", X.shape)
print("Binary labels:", y_binary.value_counts())
print("Attack types:", y_multi.value_counts())


X shape: (16318, 26)
Binary labels: Label
0    14272
1     2046
Name: count, dtype: int64
Attack types: Attack Category
normal             14272
Spoofing            1124
Data Alteration      922
Name: count, dtype: int64


In [5]:
#filling missing values

from sklearn.impute import SimpleImputer

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")


In [6]:
#filling missing values
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", num_imputer),
            ("scale", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("impute", cat_imputer),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)


In [7]:
#

from sklearn.ensemble import IsolationForest

X_normal = X[y_binary == 0]  # 0 = benign

iforest = IsolationForest(
    n_estimators=300,
    contamination=0.05,   # start here, tune later
    random_state=42,
    n_jobs=-1
)

unsup_pipe = Pipeline([
    ("preprocess", preprocess),
    ("iforest", iforest)
])

unsup_pipe.fit(X_normal)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Dport', 'SrcBytes',
                                                   'DstBytes', 'SrcLoad',
                                                   'DstLoad', 'SrcGap',
                                                   'DstGap', 'SIntPkt',
                                                   'DIntPkt', 'SIntPktAct',
                                                   'DIntPktAct', 'SrcJitter',
                                                   'DstJitter', 'sMaxPktSz',
                                                   'dMaxPktSz', 'sMinPktSz',
                                                   'dMinPktSz', 'Dur', 'Trans',
                                                   'TotPkts', 'TotBytes',
                                                   'Load', 'Rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Dir', 'Flgs', 'Sport'])])),
                ('iforest',
                 IsolationForest(contamination=0.05, n_estimators=300,
                                 n_jobs=-1, random_state=42))])

In [8]:
#calculating the anomaly score

from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

Xt = unsup_pipe.named_steps["preprocess"].transform(X)
normality = unsup_pipe.named_steps["iforest"].decision_function(Xt)

anomaly_score = (normality.max() - normality) / (normality.max() - normality.min() + 1e-12)


In [9]:
#setting the normal benchmark

normal_mask = (y_binary == 0)

def q(col, quantile):
    return X.loc[normal_mask, col].quantile(quantile)

TH = {
    "Rate_hi": q("Rate", 0.995),
    "TotBytes_hi": q("TotBytes", 0.995),
    "TotPkts_hi": q("TotPkts", 0.995),
    "Load_hi": q("Load", 0.995),
    "Dur_lo": q("Dur", 0.005),
    "Jitter_hi": max(q("SrcJitter", 0.995), q("DstJitter", 0.995)),
    "Gap_hi": max(q("SrcGap", 0.995), q("DstGap", 0.995)),
    "Asym_hi": 10.0,  # traffic asymmetry ratio
}

# Rare destination ports in normal traffic (<0.1% frequency)
freq = X.loc[normal_mask, "Dport"].value_counts(normalize=True)
RARE_DPORTS = set(freq[freq < 0.001].index)

TH, len(RARE_DPORTS)


({'Rate_hi': np.float64(660.258085),
  'TotBytes_hi': np.float64(682.0),
  'TotPkts_hi': np.float64(7.0),
  'Load_hi': np.float64(436650.96),
  'Dur_lo': np.float64(0.009087355),
  'Jitter_hi': np.float64(50.86771686000001),
  'Gap_hi': np.float64(0.0),
  'Asym_hi': 10.0},
 0)

In [10]:
#function that basically raise a flag when the features are not within the normal benchmark

def compute_rules(df: pd.DataFrame) -> pd.DataFrame:
    rules = pd.DataFrame(index=df.index)

    # Burst / volume signals
    rules["rate_burst"]  = (df["Rate"] > TH["Rate_hi"]).astype(int)
    rules["bytes_spike"] = (df["TotBytes"] > TH["TotBytes_hi"]).astype(int)
    rules["pkts_spike"]  = (df["TotPkts"] > TH["TotPkts_hi"]).astype(int)
    rules["load_spike"]  = (df["Load"] > TH["Load_hi"]).astype(int)

    # Timing instability (often present in attacks / disruptions)
    rules["gap_spike"]    = ((df["SrcGap"] > TH["Gap_hi"]) | (df["DstGap"] > TH["Gap_hi"])).astype(int)
    rules["jitter_spike"] = ((df["SrcJitter"] > TH["Jitter_hi"]) | (df["DstJitter"] > TH["Jitter_hi"])).astype(int)

    # Short duration + lots of packets (burst/flood-ish)
    rules["short_dur_many_pkts"] = ((df["Dur"] < TH["Dur_lo"]) & (df["TotPkts"] > TH["TotPkts_hi"])).astype(int)

    # Asymmetry (scan / exfil-ish)
    asym = (df["SrcBytes"] + 1.0) / (df["DstBytes"] + 1.0)
    rules["asymmetry"] = ((asym > TH["Asym_hi"]) | (asym < 1.0 / TH["Asym_hi"])).astype(int)

    # Rare destination ports (in normal)
    rules["rare_dport"] = df["Dport"].isin(RARE_DPORTS).astype(int)

    # Aggregate
    rules["rule_score"] = rules.sum(axis=1)
    rules["rules_flag"] = (rules["rule_score"] >= 1).astype(int)  # set to >=2 if too many alerts

    return rules

rules_df = compute_rules(X)
rules_df[["rule_score", "rules_flag"]].describe()


,rule_score,rules_flag
count,16318.000000,16318.000000
mean,0.043694,0.028619
std,0.287409,0.166737
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,3.000000,1.000000


In [11]:
#evaluation for the models

# IF threshold (start with 95th percentile; tune later)
anomaly_threshold = np.percentile(anomaly_score, 95)
if_flag = (anomaly_score >= anomaly_threshold).astype(int)

hybrid_flag = ((if_flag == 1) | (rules_df["rules_flag"] == 1)).astype(int)

print("=== Isolation Forest only ===")
print(classification_report(y_binary, if_flag))

print("=== Rules only ===")
print(classification_report(y_binary, rules_df["rules_flag"]))

print("=== Hybrid (IF OR Rules) ===")
print(classification_report(y_binary, hybrid_flag))

print("ROC-AUC (IF anomaly_score):", roc_auc_score(y_binary, anomaly_score))


=== Isolation Forest only ===
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     14272
           1       0.94      0.45      0.61      2046

    accuracy                           0.93     16318
   macro avg       0.94      0.72      0.79     16318
weighted avg       0.93      0.93      0.92     16318

=== Rules only ===
              precision    recall  f1-score   support

           0       0.89      0.99      0.94     14272
           1       0.69      0.16      0.26      2046

    accuracy                           0.89     16318
   macro avg       0.79      0.57      0.60     16318
weighted avg       0.87      0.89      0.85     16318

=== Hybrid (IF OR Rules) ===
              precision    recall  f1-score   support

           0       0.93      0.99      0.96     14272
           1       0.86      0.45      0.59      2046

    accuracy                           0.92     16318
   macro avg       0.89      0.72      0.77     16

In [12]:
from sklearn.metrics import precision_score, recall_score

for pct in [90, 92, 95, 97, 98, 99]:
    thr = np.percentile(anomaly_score, pct)
    if_flag_p = (anomaly_score >= thr).astype(int)
    hybrid_p = ((if_flag_p == 1) | (rules_df["rules_flag"] == 1)).astype(int)

    p = precision_score(y_binary, hybrid_p, zero_division=0)
    r = recall_score(y_binary, hybrid_p, zero_division=0)
    print(f"pct={pct:>2}  hybrid_precision={p:.3f}  hybrid_recall={r:.3f}")


pct=90  hybrid_precision=0.555  hybrid_recall=0.462
pct=92  hybrid_precision=0.681  hybrid_recall=0.459
pct=95  hybrid_precision=0.863  hybrid_recall=0.451
pct=97  hybrid_precision=0.778  hybrid_recall=0.251
pct=98  hybrid_precision=0.693  hybrid_recall=0.161
pct=99  hybrid_precision=0.687  hybrid_recall=0.157


In [13]:
#supervised section
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)


In [14]:
#function for evaluation

from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(model_pipe, X_test, y_test, proba_positive=True):
    y_pred = model_pipe.predict(X_test)
    print(classification_report(y_test, y_pred))

    # ROC-AUC if model supports predict_proba or decision_function
    if hasattr(model_pipe, "predict_proba"):
        y_score = model_pipe.predict_proba(X_test)[:, 1]
        print("ROC-AUC:", roc_auc_score(y_test, y_score))
    elif hasattr(model_pipe, "decision_function"):
        y_score = model_pipe.decision_function(X_test)
        print("ROC-AUC:", roc_auc_score(y_test, y_score))
    else:
        print("ROC-AUC: (not available — model has no score output)")


In [15]:
#random forest model

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", rf)
])

rf_pipe.fit(X_train, y_train)

print("=== Random Forest ===")
evaluate_model(rf_pipe, X_test, y_test)


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2855
           1       1.00      0.58      0.73       409

    accuracy                           0.95      3264
   macro avg       0.97      0.79      0.85      3264
weighted avg       0.95      0.95      0.94      3264

ROC-AUC: 0.8815465511113774


In [16]:
import joblib

# Save final models for 
unsup_pipe.fit(X)
joblib.dump(unsup_pipe, "iforest_pipe.joblib")
joblib.dump(rf_pipe, "rf_pipe.joblib")
joblib.dump(list(X.columns), "features.joblib")

print("Final models frozen and saved.")


Final models frozen and saved.


In [17]:
import joblib
import pandas as pd

# Load frozen artifacts
if_pipe = joblib.load("iforest_pipe.joblib")
rf_pipe = joblib.load("rf_pipe.joblib")
FEATURES = joblib.load("features.joblib")

def score_one(flow: dict, attack_threshold: float = 0.5):
    # Build 1-row dataframe in the same feature order used in training
    row = {k: flow.get(k, None) for k in FEATURES}
    X_one = pd.DataFrame([row], columns=FEATURES)

    # Stage 1: Isolation Forest (1=inlier, -1=outlier)
    is_anomalous = (int(if_pipe.predict(X_one)[0]) == -1)

    out = {"is_anomalous": is_anomalous}

    # Optional anomaly score
    if hasattr(if_pipe, "score_samples"):
        out["anomaly_score"] = float(if_pipe.score_samples(X_one)[0])
    elif hasattr(if_pipe, "decision_function"):
        out["anomaly_score"] = float(if_pipe.decision_function(X_one)[0])
    else:
        out["anomaly_score"] = None

    # Stage 2: Random Forest only if anomalous
    if is_anomalous:
        if hasattr(rf_pipe, "predict_proba"):
            p = float(rf_pipe.predict_proba(X_one)[0][1])
            out["attack_probability"] = p
            out["is_attack"] = (p >= attack_threshold)
        else:
            pred = int(rf_pipe.predict(X_one)[0])
            out["is_attack"] = (pred == 1)
            out["attack_probability"] = None
    else:
        out["is_attack"] = False
        out["attack_probability"] = 0.0

    return out


In [18]:
#Extratrees model, boost
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import Pipeline

et = ExtraTreesClassifier(
    n_estimators=700,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

et_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", et)
])

et_pipe.fit(X_train, y_train)

print("=== ExtraTrees ===")
evaluate_model(et_pipe, X_test, y_test)


=== ExtraTrees ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2855
           1       1.00      0.58      0.73       409

    accuracy                           0.95      3264
   macro avg       0.97      0.79      0.85      3264
weighted avg       0.95      0.95      0.94      3264

ROC-AUC: 0.8849288555658797


In [19]:
#evaluation for extratrees

try:
    from xgboost import XGBClassifier
    from sklearn.pipeline import Pipeline

    xgb = XGBClassifier(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    )

    xgb_pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", xgb)
    ])

    xgb_pipe.fit(X_train, y_train)

    print("=== XGBoost ===")
    evaluate_model(xgb_pipe, X_test, y_test)

except Exception as e:
    print("XGBoost not available:", e)



=== XGBoost ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2855
           1       0.96      0.59      0.73       409

    accuracy                           0.95      3264
   macro avg       0.95      0.80      0.85      3264
weighted avg       0.95      0.95      0.94      3264

ROC-AUC: 0.894997837620269
